# WhatsApp Chatbot Development
Tutorial: https://www.youtube.com/watch?v=3YPeh-3AFmM

Github: https://github.com/daveebbelaar/python-whatsapp-bot

In [1]:
import json
import os
import requests
from dotenv import load_dotenv
import logging

from typing import Union
from pathlib import Path

from src.utils.chatbot_app import create_app

In [2]:
if not load_dotenv("../private/chatbot.env"): raise ValueError("Failed to load .env file")

# important variables:
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")
RECIPIENT_WAID = os.getenv("RECIPIENT_WAID")
PHONE_NUMBER_ID = os.getenv("PHONE_NUMBER_ID")
VERSION = os.getenv("VERSION")

APP_ID = os.getenv("APP_ID")
APP_SECRET = os.getenv("APP_SECRET")

VERIFY_TOKEN = os.getenv("VERIFY_TOKEN") # from webhook's callback URL (ngrok) in meta developers

In [6]:
class WhatsAppChatbot:
    """
    Class WhatsAppChatbot
    Manages sending messages via WhatsApp using the Facebook Graph API. It loads necessary configuration from an environment file and provides methods to send text messages.

    Parameters
    ----------
    chatbot_env_path : Union[str, Path]
        Path to the environment file (.env) containing necessary API credentials and configuration.
    verbose : bool, optional
        If True, enables verbose logging for debugging. Default is False.

    Raises
    ------
    ValueError
        If loading the environment file fails.

    Attributes
    ----------
    _access_token : str
        Access token required for authenticating API requests.
    _recipient_waid : str
        WhatsApp ID of the message recipient.
    _phone_number_id : str
        Phone number ID linked with the WhatsApp Business API.
    _version : str
        API version to be used for the requests.
    _app_id : str
        Application ID for the API integration.
    _app_secret : str
        Application secret for authentication.
    _verify_token : str
        Token used to verify webhook integration with the API.
    verbose : bool
        Indicates whether verbose logging is enabled.
    """
    def __init__(self, chatbot_env_path: Union[str, Path], verbose: bool = False):
        if not load_dotenv(chatbot_env_path): raise ValueError("Failed to load .env file")
        self._access_token = os.getenv("ACCESS_TOKEN")
        self._recipient_waid = os.getenv("RECIPIENT_WAID")
        self._phone_number_id = os.getenv("PHONE_NUMBER_ID")
        self._version = os.getenv("VERSION")
        self._app_id = os.getenv("APP_ID")
        self._app_secret = os.getenv("APP_SECRET")
        self._verify_token = os.getenv("VERIFY_TOKEN")  # from webhook's callback URL (ngrok) in meta developers
        
        self.verbose = verbose

    def __call__(self, message: str):
        """
        Parameters
        ----------
        message : str
            The message to be sent using the `send_message` method.
        """
        self.send_message(message)

    def send_message(self, message: str):
        """
        Sends a text message to a specified recipient.

        Parameters
        ----------
        message : str
            The text content of the message to be sent.
        """
        data = self._get_text_message_input(recipient=RECIPIENT_WAID, text=message)
        self._send_message_backend(data, verbose=self.verbose)

    # auxiliary methods:
    @staticmethod
    def _get_text_message_input(recipient, text):
        """
        Prepare data for _send_message method.
        
        Parameters
        ----------
        recipient : str
            The phone number or unique identifier of the message recipient.
        text : str
            The body of the text message to be sent.
        """
        return json.dumps(
            {
                "messaging_product": "whatsapp",
                "recipient_type": "individual",
                "to": recipient,
                "type": "text",
                "text": {"preview_url": False, "body": text},
            }
        )

    @staticmethod
    def _send_message_backend(data, verbose=False):
        """
        Sends a message using the Facebook Graph API.

        Parameters
        ----------
        data : dict
            The JSON payload to be sent in the HTTP POST request body.
        verbose : bool, optional
            If True, prints the status code, response headers, and response body for debugging. Default is False.

        Returns
        -------
        response : requests.Response
            The HTTP response object returned by the API call.
        """
        headers = {
            "Content-type": "application/json",
            "Authorization": f"Bearer {ACCESS_TOKEN}",
        }

        url = f"https://graph.facebook.com/{VERSION}/{PHONE_NUMBER_ID}/messages"

        response = requests.post(url, data=data, headers=headers)
        if response.status_code == 200:
            if verbose:
                print("Status:", response.status_code)
                print("Content-type:", response.headers["content-type"])
                print("Body:", response.text)
            return response
        else:
            if verbose:
                print(response.status_code)
                print(response.text)
            return response
        
    def test_connection(self):
        url = f"https://graph.facebook.com/{self._version}/{self._phone_number_id}/messages"
        headers = {
            "Authorization": "Bearer " + self._access_token,
            "Content-Type": "application/json",
        }
        data = {
            "messaging_product": "whatsapp",
            "to": self._recipient_waid,
            "type": "template",
            "template": {"name": "hello_world", "language": {"code": "en_US"}},
        }
        response = requests.post(url, headers=headers, json=data)
        if self.verbose:
            print(response.status_code)
            print(response.text)

In [7]:
chatter = WhatsAppChatbot("../private/chatbot.env", verbose=True)

In [8]:
chatter.test_connection()

200
{"messaging_product":"whatsapp","contacts":[{"input":"+4915257468736","wa_id":"4915257468736"}],"messages":[{"id":"wamid.HBgNNDkxNTI1NzQ2ODczNhUCABEYEjU5NzE3MkYzRTQ2NzQ3MjNBQgA=","message_status":"accepted"}]}


In [9]:
chatter('Hello bello')

Status: 200
Content-type: application/json; charset=UTF-8
Body: {"messaging_product":"whatsapp","contacts":[{"input":"+4915257468736","wa_id":"4915257468736"}],"messages":[{"id":"wamid.HBgNNDkxNTI1NzQ2ODczNhUCABEYEjY1Rjg2RDI2QzFENjYxN0MwRgA="}]}


## Receive Messages

In [ ]:
# webhook setup with ngrok

# run in terminal:
#!ngrok http 8000 --domain prompt-crayfish-cunning.ngrok-free.app

# callback URL: https://prompt-crayfish-cunning.ngrok-free.app/webhook
# verification token from developers.facebook.com needs to match env

In [ ]:

# this and ngrok in terminal need to remain running:

app = create_app()

if __name__ == "__main__":
    logging.info("Flask app started")
    app.run(host="0.0.0.0", port=8000)
    
# __init__.py -> register_blueprint(webhook_blueprint) -> view.py -> handle_message -> whatsapp_utils.py -> process_whatsapp_message() -> generate_response() 

## AI?

In [ ]:
# see services -> openai_service.py -> generate_response()